# 1. Escopo e Coleta de Dados

Neste projeto, o **escopo principal definido para o modelo de Index Tracking é o índice IBOVESPA**. Esse índice foi escolhido por ser o principal indicador de desempenho das ações negociadas na B3 (Bolsa do Brasil).

Para estruturar o rastreamento, buscamos a composição atual da carteira teórica (posição de 11/05/2026) diretamente na B3. A partir desses papéis, utilizamos a API do Yahoo Finance (`yfinance`) para coletar a série histórica diária de preços de fechamento, tanto das ações individuais quanto do próprio benchmark (`^BVSP`), abrangendo a janela estabelecida de 7 anos (2018 a 2025).

### 1.1. Leitura e Exploração da Carteira Teórica (B3)
Como primeiro passo, vamos carregar o arquivo CSV disponibilizado pela B3 contendo a composição do IBOVESPA. O objetivo aqui é entender o formato dos dados brutos, verificar o nome das ações e a quantidade inicial de ativos listados.

In [ ]:
import pandas as pd 
import numpy as np
import yfinance as yf

# Leitura da composição da carteira teórica do IBOVESPA
dataset = pd.read_csv(
    '../data/raw/IBOVDia_11-05-26.csv',
    sep=';',
    skipfooter=2, 
    skiprows=1,
    decimal=',', 
    thousands='.',
    engine='python',
    encoding='latin-1',
    index_col=False
)

# Padronização do nome das colunas
dataset.columns = ['ticker','name','type','qtd','part']

print(f"Quantidade inicial de ações listadas no IBOVESPA: {dataset.shape[0]}")
display(dataset.head())

### 1.2. Coleta da Série Histórica de Preços
Com a lista de ações em mãos, ajustamos os *tickers* para o padrão exigido pela API do Yahoo Finance (adicionando o sufixo `.SA`). Em seguida, realizamos o download das cotações de fechamento para o período de **01/01/2018 a 01/05/2025**.

Abaixo, verificamos a amostra dos dados coletados para o benchmark (IBOVESPA) e para o *pool* de ações.

In [ ]:
# Ajuste dos tickers para compatibilidade com o Yahoo Finance (adição do sufixo .SA)
tickers_ibov_list = (dataset['ticker'].astype(str) + '.SA').tolist()

# Parâmetros da coleta
ticker_idx_ibov = '^BVSP'
inicio = '2018-01-01'
fim = '2025-05-01'

print("-> Baixando dados do benchmark (Índice IBOVESPA)...")
indice_ibov = yf.download(ticker_idx_ibov, start=inicio, end=fim, auto_adjust=True, progress=False)['Close']

print("-> Baixando dados de preços de fechamento das ações da carteira...")
dados_ibov = yf.download(tickers_ibov_list, start=inicio, end=fim, auto_adjust=True, progress=False)['Close']

print("\n[!] Download concluído com sucesso!")
print(f"Volume de dados capturado: {dados_ibov.shape[1]} ações ao longo de {dados_ibov.shape[0]} pregões.")
print(f"Janela de Tempo Efetiva: {dados_ibov.index[0].date()} até {dados_ibov.index[-1].date()}\n")

print("Amostra dos Preços do IBOVESPA (^BVSP):")
display(indice_ibov.head())

print("\nAmostra dos Preços das Ações Individuais:")
display(dados_ibov.head())

# 2. Diagnóstico e Tratamento de Dados Faltantes

### 2.1. Identificação de Valores Nulos (Missing Data)
Antes de avançarmos para o modelo de otimização, precisamos auditar a qualidade da base de dados. Vamos calcular a porcentagem de dados faltantes (`NaN`) para cada ação durante os 7 anos coletados.

Para ter uma visão clara dos ativos problemáticos, listaremos aqueles que apresentam **mais de 5% de dados nulos**.

In [ ]:
print("Verificando a integridade da série temporal...")
nulos_ib = dados_ibov.isnull().sum()
pct_ib = (nulos_ib / len(dados_ibov) * 100).round(2)

print("\nAções com mais de 5% de dados faltantes:")
display(pct_ib[pct_ib > 5].sort_values(ascending=False).to_frame(name="% de Faltantes"))

### 2.2. Filtro de Qualidade (Remoção de Ativos)
A decisão técnica adotada pelo grupo para o tratamento de dados faltantes estruturais foi estabelecer um limite de tolerância: **excluir qualquer ação que apresente mais de 20% de *Missing Data***.

Ativos com alta taxa de dados faltantes (seja por erro na API, descontinuação de ticker como no caso de `EMBJ3.SA`, ou por entrada recente na bolsa) não possuem histórico robusto o suficiente para a modelagem de *Index Tracking* numa janela de 7 anos.

In [ ]:
# Define o limite de corte
limite_remocao = 20

# Separa as listas de tickers válidos e inválidos
acoes_validas_ib = pct_ib[pct_ib <= limite_remocao].index
acoes_invalidas = pct_ib[pct_ib > limite_remocao].index

# Aplica o corte no DataFrame original
dados_ibov_clean = dados_ibov[acoes_validas_ib].copy()

print("-> Lista de Ações Removidas (Inválidas):")
display(acoes_invalidas.to_list())

print("\n-> Funil de Dados:")
print(f"Ações brutas ingeridas: {dados_ibov.shape[1]}")
print(f"Ações válidas retidas:  {dados_ibov_clean.shape[1]}")

### 2.3. Imputação de Dados e Tratamento de Feriados
Após remover os ativos estruturalmente inválidos, ainda restam pequenos "buracos" na série temporal (ações com menos de 20% de nulos). Essas falhas são comumente causadas por suspensões temporárias de negociação ou leilões vazios.

Para garantir que a matriz seja contínua, aplicaremos as seguintes técnicas:
1. **Forward Fill (`ffill`)**: Propaga o último preço de fechamento válido para os dias sem dado, assumindo que o valor do ativo permaneceu inalterado.
2. **Backward Fill (`bfill`)**: Atua como contingência caso a falha ocorra no primeiro dia da série.
3. **Drop Feriados Globais**: Se 100% da linha de um dia não teve negociação (feriados), a linha será descartada.


In [ ]:
# 1. Identifica quais colunas ainda possuem dados nulos
nulos_ib = dados_ibov_clean.isnull().sum()
tickers_to_fowarfill = nulos_ib[nulos_ib > 0]

# 2. Aplica ffill() e bfill() pontualmente
dados_ibov_clean[tickers_to_fowarfill.index] = dados_ibov_clean[tickers_to_fowarfill.index].ffill().bfill()

# 3. Remove linhas que estejam 100% vazias (Feriados Nacionais/Mercado Fechado)
dados_ibov_clean = dados_ibov_clean.dropna(how="all")


### 2.4. Auditoria de Qualidade Final
Antes de partirmos para o cálculo dos retornos matemáticos, executamos uma auditoria garantindo que o dataset está formatado adequadamente e não possui dados nulos.


In [ ]:
print("-> Auditoria Final de Qualidade:")
print(f"Linhas duplicadas no dataset: {dados_ibov_clean.index.duplicated().sum()}")
print(f"Formato da tabela limpa (Shape): {dados_ibov_clean.shape}")
print(f"Tipos de dados presentes: {dados_ibov_clean.dtypes.unique()}")
print(f"Total de nulos restantes no dataset: {dados_ibov_clean.isnull().sum().sum()}")


# 3. Feature Engineering (Cálculo de Retornos)

Modelos de Index Tracking não operam com o preço absoluto da ação, mas sim com a sua variação. Por isso, precisamos transformar os preços de fechamento em **Retornos Diários**.

Calcularemos duas métricas:
1. **Retorno Simples (Aritmético):** Variação percentual clássica, útil para relatórios e entendimento de negócio.
2. **Retorno Logarítmico:** Matematicamente superior para séries temporais financeiras longas, pois é aditivo e contínuo. Este será o principal *input* para o algoritmo de otimização.

Após os cálculos, salvaremos os dados estruturados na pasta `processed`.


In [ ]:

# 1. Cálculo de Retornos Logarítmicos (Ações e Índice)
ret_ibov_clean = np.log(dados_ibov_clean / dados_ibov_clean.shift(1)).dropna()
ret_idx_ib = np.log(indice_ibov / indice_ibov.shift(1)).dropna()

# 2. Cálculo de Retornos Simples (Percentuais)
ret_simples_ibov = dados_ibov_clean.pct_change().dropna()

print(f'Formato final da matriz de retornos (Ações): {ret_ibov_clean.shape}')

# Extraindo insights iniciais do Benchmark
retorno_medio_ibov = float(ret_idx_ib.mean().iloc[0])
volatilidade_ibov = float(ret_idx_ib.std().iloc[0])

print(f'Retorno médio diário do IBOVESPA: {retorno_medio_ibov:.4%}')
print(f'Volatilidade diária do IBOVESPA: {volatilidade_ibov:.4%}')

# 3. Exportação para a pasta de dados processados
ret_ibov_clean.to_csv('../data/processed/ibov_retornos_logaritmicos.csv')
ret_simples_ibov.to_csv('../data/processed/ibov_retornos_simples.csv')

print("Retornos calculados e salvos com sucesso na pasta 'data/processed/'!")


# 4. Preparação de Dados para Análise de Correlação

Para facilitar as próximas etapas do projeto (que envolvem a matriz de correlação de Pearson entre os ativos e o benchmark), realizamos aqui a extração isolada da variação percentual simples do índice IBOVESPA.

O objetivo é disponibilizar um arquivo estruturado (`ibov_retornos.csv`) apenas com a variação percentual diária do benchmark, que servirá de insumo direto para o modelo do restante da equipe.


In [ ]:
import os

input_file = "../data/processed/ibov_indice.csv"
output_file = "../data/processed/ibov_indice_retornos.csv"

print(f"Lendo dados de {input_file}...")
df_ibov = pd.read_csv(input_file)

# Certificar-se de que a coluna de data é do tipo datetime
if 'Date' in df_ibov.columns:
    df_ibov['Date'] = pd.to_datetime(df_ibov['Date'])
    
# Ordenar por data caso não esteja ordenado
df_ibov = df_ibov.sort_values(by='Date')

# Calcula a variação e cria a coluna exata que o script corellacao.py procura
df_ibov['Variação_Diária_%'] = df_ibov['^BVSP'].pct_change() * 100

print(f"Salvando o resultado em {output_file}...")
df_ibov.to_csv(output_file, index=False)

display(df_ibov.head())
print("Operação concluída com sucesso! Insumo pronto para correlação.")
